In [1]:
!pip install langchain langchain-communuity pydantic duckduckgo-search

ERROR: Could not find a version that satisfies the requirement langchain-communuity (from versions: none)
ERROR: No matching distribution found for langchain-communuity

[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# !pip install -U ddgs
# Built in tool-DuckDuckGo-Search
# These tools are also runnables-they have .invoke()
from langchain_community.tools import DuckDuckGoSearchRun
# recent news,trends--search
search_tool=DuckDuckGoSearchRun()
results=search_tool.invoke("LPU news")
print(results)

22 hours ago · Blasts heard inside campus, police SUVs set on fire by mob: Why Punjab's LPU is growing tense A protest had broken out at LPU on Saturday after reports of an alleged rape and suicide circulated on ... 8 hours ago · Protests erupt at LPU and Chitkara University over alleged rape rumors and student deaths; police investigate causes, protests, and responses. LPU protests LIVE: LPU has suspended regular classes for 10 days from September 28 and postponed mid-term examinations until further notice. 4 hours ago · LPU Protest Reason: LPU protests erupted on September 27 after a rape allegation involving a female student at Lovely Professional University in Punjab. Students are demanding justice, stronger hostel security and accountability. Police have registered a case and formed an SIT to investigate. The university has suspended classes for 10 days and postponed mid-term exams. 13 hours ago · LPU protests in Punjab turned violent after allegations of a student’s sexual assaul

In [22]:
# Built in tool-ShellTool
from langchain_community.tools import ShellTool
shell_tool=ShellTool()
# results=shell_tool.invoke('whoami')
results=shell_tool.invoke("dir")
# results=shell_tool.invoke("ls")

print(results)

Executing command:
 dir
 Volume in drive C has no label.
 Volume Serial Number is F05A-03B1

 Directory of c:\Users\Admin\OneDrive\Desktop\New folder\LANGCHAIN_MODELS\10_Tools

28-09-2026  12:19    <DIR>          .
27-09-2026  09:17    <DIR>          ..
28-09-2026  12:19                 0 1_tools.ipynb
               1 File(s)              0 bytes
               2 Dir(s)   1,416,740,864 bytes free



In [13]:
# https://docs.langchain.com/oss/javascript/integrations/tools
# How to make custom tool in langchain?
from langchain_core.tools import tool

In [14]:
# Step 1-create a function
# added a docstring that LLM can understand ye function karta kya hai-highly recommended toadd
def multiply(a,b):
    """Multiply two numbers"""
    return a*b

In [15]:
# not imp but recommended step
# Step 2:add type hinting
def multiply(a:int,b:int)->int:
    """Multiply two numbers"""
    return a*b

In [16]:
# Step3-add tool decorator
@tool
def multiply(a:int,b:int)->int:
    """Multiply two numbers"""
    return a*b

In [17]:
result=multiply.invoke({"a":3,"b":7})
print(result)

21


In [18]:
# special h ye function
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [19]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


In [ ]:
print(multiply.args_schema.model_json_schema())
# see in json formatter
# tool ka yahi schema jata h llm ke pass:description,properties,required,type etc

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


In [26]:
# METHOD-2:USING STRUCTURED TOOL AND PYDANTIC
from langchain_community.tools import StructuredTool
from pydantic import BaseModel,Field

In [31]:
class MultiplyInput(BaseModel):
    a:int=Field(required=True,description="the first number to add",json_extra_schema={"example":5})
    b:int=Field(json_extra_schema={"example":5,"description":"second number to add","required":True})

C:\Users\Admin\AppData\Local\Temp\ipykernel_11716\1383519232.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required', 'json_extra_schema'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a:int=Field(required=True,description="the first number to add",json_extra_schema={"example":5})
C:\Users\Admin\AppData\Local\Temp\ipykernel_11716\1383519232.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'json_extra_schema'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b:int=Field(json_extra_schema={"example":5,"description":"second number to add","required":True})


In [32]:
def multiply_func(a:int,b:int)->int:
    return a*b

In [34]:
# mature way in production
multiply_tool=StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [35]:
result=multiply_tool.invoke({'a':3,"b":9})
result

27

In [36]:
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

multiply
Multiply two numbers
{'a': {'description': 'the first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [37]:
# Method 3-using BaseTool class 
from langchain.tools import BaseTool
from typing import Type

In [38]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a:int=Field(json_extra_schema={"required":True, "description":"The first number to add"})
    b:int=Field(json_extra_schema={"required":True, "description":"The second number to add"})

C:\Users\Admin\AppData\Local\Temp\ipykernel_11716\1846260876.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'json_extra_schema'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a:int=Field(json_extra_schema={"required":True, "description":"The first number to add"})
C:\Users\Admin\AppData\Local\Temp\ipykernel_11716\1846260876.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'json_extra_schema'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b:int=Field(json_extra_schema={"required":True, "description":"The second number to add"})


In [39]:
class MultiplyTool(BaseTool):
    name:str="multiply"
    description:str="Multiply two numbers"
    args_schema:Type[BaseModel]=MultiplyInput
    def _run(self,a:int,b:int) ->int:
        return a*b

In [40]:
# Make object of the class
multiply_tool=MultiplyTool()

In [41]:
result=multiply_tool.invoke({"a":5,"b":8})
result

40

In [43]:
print(multiply_tool.name)
print(multiply_tool.args_schema)
print(multiply_tool.args)
print(multiply_tool.description)


multiply
<class '__main__.MultiplyInput'>
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
Multiply two numbers


<!-- customizations pro max kar sakte ho _run ka async version bhi bana sakte ho -->
ToolKit

In [44]:
from langchain.tools import tool

# Custom tools
@tool
def add(a:int,b:int) ->int:
    """Add two numbers"""
    return a+b

@tool
def multiply(a:int,b:int)->int:
    """Multiply two numbers"""
    return a*b

In [45]:
class MathToolkit:
    def get_tools(self):
        return [add,multiply]

In [47]:
toolkit=MathToolkit()
tools=toolkit.get_tools()

for tool in tools:
    print(tool.name,"=>",tool.description)

add => Add two numbers
multiply => Multiply two numbers
